# E1: class-balanced training on Colab GPU

Trains the E1 configuration (class-weighted loss, stronger train-only augmentation, 20 epochs, checkpoint chosen by class-balanced validation NLL) for the five seeds. An optional E0 control run on the same GPU type can be switched on in cell 1. Each finished run is copied to Drive, so a disconnect loses at most the run in progress; re-running the notebook skips finished runs.

**Before you start**

1. Runtime > Change runtime type > **GPU** (T4 is fine; use the same GPU type for every seed).
2. Upload `skin-disease-app-e1.bundle` into the `ham10000_e0` folder in Drive.
3. Choose **Runtime > Run all**, and keep this tab open.


In [ ]:
# 1. Settings and Drive
import json, hashlib, os, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

BASE = Path('/content/drive/MyDrive/ham10000_e0')
BUNDLE = BASE / 'skin-disease-app-e1.bundle'
RESULTS = BASE / 'e1_results'
REPO = Path('/content/skin-disease-app-e1')
PY = '/content/venv/bin/python'
E1_SEEDS = [42, 123, 2024, 3407, 5001]
RUN_E0_CONTROL = False   # set True to also rerun E0 (30 epochs) on this GPU, seed 42
E0_CONTROL_SEEDS = [42]

os.environ['MPLBACKEND'] = 'Agg'  # Colab's inline plot setting breaks matplotlib in the separate environment

assert BUNDLE.is_file(), f'Upload the bundle first: {BUNDLE}'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout or 'NO GPU: change the runtime type')


def run(cmd, cwd=REPO, log=None):
    """Runs a command, streaming its output; stops the notebook if it fails."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    sink = open(log, 'a', encoding='utf-8') if log else None
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
        if sink:
            sink.write(line)
    code_ = p.wait()
    if sink:
        sink.close()
    if code_:
        raise RuntimeError(f'Command failed with exit code {code_}: {" ".join(cmd)}')

In [ ]:
# 2. Get the exact committed code. Each run records this commit.
if not REPO.exists():
    run(['git', 'clone', '-b', 'e1-class-balanced', BUNDLE, REPO], cwd='/content')
run(['git', 'log', '-1', '--format=%h %s'])
run(['git', 'status', '--short'])

In [ ]:
# 3. Install the pinned versions in a separate Python 3.11 environment (a few minutes)
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], cwd='/content')
run(['uv', 'venv', '--python', '3.11', '--allow-existing', '/content/venv'], cwd='/content')
run(['uv', 'pip', 'install', '-q', '--python', PY, '-r', 'backend/requirements.txt', '-r', 'research/requirements.txt'])
run([PY, '-c', "import torch, torchvision; print('torch', torch.__version__, 'vision', torchvision.__version__, 'cuda', torch.cuda.is_available()); assert torch.cuda.is_available(), 'GPU not visible'"])

In [ ]:
# 4. Get HAM10000 and validate it against the frozen split
run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], cwd='/content')
import kagglehub
DATA_ROOT = next(Path(kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')).rglob('HAM10000_metadata.csv')).parent
print('data root:', DATA_ROOT)
run([PY, '-m', 'research.data.split', '--config', 'research/configs/e0.yaml', '--data-root', DATA_ROOT, '--check'])

In [ ]:
# 5. Train. Check the first epoch's time in the output.
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


def finished(run_dir):
    meta = run_dir / 'run_metadata.json'
    return (meta.is_file() and json.loads(meta.read_text())['status'] == 'complete'
            and (run_dir / 'predictions' / 'test.npz').is_file())


JOBS = [('e1', s) for s in E1_SEEDS] + ([('e0', s) for s in E0_CONTROL_SEEDS] if RUN_E0_CONTROL else [])
for exp, seed in JOBS:
    out_name = 'E1' if exp == 'e1' else 'E0'
    dst = RESULTS / out_name / f'seed_{seed}'
    if finished(dst):
        print(f'{exp} seed {seed} is already on Drive; skipping')
        continue
    start = time.time()
    run_dir = REPO / 'research/outputs' / out_name / f'seed_{seed}'
    log = f'/content/train_{exp}_{seed}.log'
    if not finished(run_dir):
        shutil.rmtree(run_dir, ignore_errors=True)
        Path(log).unlink(missing_ok=True)
        run([PY, '-u', '-m', 'research.training.train', '--config', f'research/configs/{exp}.yaml', '--seed', seed,
             '--data-root', DATA_ROOT, '--device', 'cuda', '--num-workers', 2], log=log)
    shutil.copy(log, run_dir / 'train.log')

    shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(run_dir, dst)
    expected = json.loads((run_dir / 'run_metadata.json').read_text())['checkpoint_sha256']
    assert sha256(dst / 'best.pt') == expected, 'Drive copy of best.pt differs'
    print(f'{exp} seed {seed} finished in {(time.time() - start) / 60:.0f} minutes and saved to Drive', flush=True)


In [ ]:
# 6. Quick summary. The full evaluation runs locally.
import numpy as np

CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
for exp, seed in JOBS:
    run_dir = RESULTS / ('E1' if exp == 'e1' else 'E0') / f'seed_{seed}'
    meta = json.loads((run_dir / 'run_metadata.json').read_text())
    z = np.load(run_dir / 'predictions' / 'test.npz')
    y, pred = z['labels'], z['preds']
    recalls = [(pred[y == k] == k).mean() for k in range(7)]
    print(f"{exp} seed {seed}: selected epoch {meta['selected_epoch']} of {meta['total_epochs_run']} | "
          f"test accuracy {(pred == y).mean():.4f} | balanced {np.mean(recalls):.4f} | "
          f"melanoma recall {recalls[CLASSES.index('mel')]:.4f} | device {meta['runtime']['device']} | "
          f"git {meta['environment']['git_commit'][:7]} dirty={meta['environment']['git_dirty']}")
    if meta['selected_epoch'] == meta['total_epochs_run']:
        print('  NOTE: the selected epoch is the last one run; report this in the paper.')


## Next step

Download the `ham10000_e0/e1_results` folder from Drive, extract it into `D:\skin-disease-app (1)\`, and send the output of cell 6.
